# Notebook 06 — t-Tests

**Module:** 03 — Statistics and Probability  
**Notebook:** 06 of 18 — Track A core  
**Prerequisites:** NB05  
**Time estimate:** 90 minutes

> **Track A note:** This is the most-asked hypothesis test in bioinformatics interviews.
> Be ready to explain the t-statistic formula, degrees of freedom, and the difference
> between paired and unpaired tests cold, without prompting.

---
## Step 1 — Motivation

The t-test answers the most common statistical question in biology: 'are these two
groups different?' Is gene X expressed differently in tumour vs. normal? Did treatment
change the protein level? Does the drug reduce viral load? The t-test applies when:
the data are approximately Normal (or n is large enough for CLT), and the sample sizes
are small enough that you cannot assume the population variance is known.

---
## Step 2 — Intuition

The t-statistic measures how many standard errors the observed difference is away from
zero. A large |t| means the observed difference is unlikely under H₀ (small p-value).

$$t = \frac{\text{signal}}{\text{noise}} = \frac{\bar{x}_1 - \bar{x}_2}{\text{SE of the difference}}$$

The SE shrinks with larger sample size — which is why more samples gives smaller p-values
for the same true difference.

---
## Step 3 — Biological Background

**When to use each t-test variant:**

| Variant | When | Biological example |
|---------|------|-------------------|
| One-sample | Compare one group to a known reference | Is mean GC content ≠ 50%? |
| Two-sample (Welch's) | Two independent groups, unequal variance | Tumour vs. normal expression |
| Two-sample (Student's) | Two independent groups, equal variance | Two knockout strains vs. wild type |
| Paired | Same subjects measured twice | Before/after drug treatment; matched samples |

**Welch's t-test** (unequal variance) is the safe default for two-sample comparisons.
Student's t-test (equal variance assumed) is slightly more powerful but fails when
variances differ — a common situation in biology.

**Degrees of freedom:**
- One-sample: $df = n - 1$
- Two-sample equal variance: $df = n_1 + n_2 - 2$
- Welch's: Welch-Satterthwaite approximation (messy formula; `scipy` handles it)

---
## Step 4 — Mathematical Explanation

**One-sample t-statistic:**
$$t = \frac{\bar{x} - \mu_0}{s / \sqrt{n}}, \quad df = n-1$$

**Two-sample (equal variance) t-statistic:**
$$t = \frac{\bar{x}_1 - \bar{x}_2}{s_p \sqrt{1/n_1 + 1/n_2}}$$
$$s_p^2 = \frac{(n_1-1)s_1^2 + (n_2-1)s_2^2}{n_1+n_2-2} \quad (\text{pooled variance})$$

**Welch's two-sample t-statistic:**
$$t = \frac{\bar{x}_1 - \bar{x}_2}{\sqrt{s_1^2/n_1 + s_2^2/n_2}}$$

**Paired t-statistic:** let $d_i = x_{1i} - x_{2i}$, then:
$$t = \frac{\bar{d}}{s_d / \sqrt{n}}, \quad df = n-1$$

p-value from the t-distribution with df degrees of freedom:
$p = 2 \cdot P(T > |t|)$ (two-tailed)

---
## Step 5 — Computational Explanation

**Implementation strategy:** build each test from the formula, validate against
`scipy.stats`. The formula implementation is the interview answer; scipy is production code.

---
## Step 6 — Python Implementation

In [ ]:
import numpy as np
import scipy.stats as stats
import matplotlib.pyplot as plt

In [ ]:
# Cell 6.1 — t-tests from scratch: all three variants

def one_sample_t(x: np.ndarray, mu0: float = 0.0) -> tuple:
    """One-sample t-test: H0: mean(x) = mu0. Returns (t_stat, p_value)."""
    n = len(x)
    t_stat = (x.mean() - mu0) / (x.std(ddof=1) / np.sqrt(n))
    p_value = 2 * stats.t.sf(abs(t_stat), df=n-1)  # two-tailed
    return t_stat, p_value

def welch_t(x: np.ndarray, y: np.ndarray) -> tuple:
    """Welch's two-sample t-test (unequal variance). Returns (t_stat, p_value)."""
    n1, n2 = len(x), len(y)
    s1_sq, s2_sq = x.var(ddof=1), y.var(ddof=1)
    se = np.sqrt(s1_sq/n1 + s2_sq/n2)
    t_stat = (x.mean() - y.mean()) / se
    # Welch-Satterthwaite degrees of freedom
    df = (s1_sq/n1 + s2_sq/n2)**2 / ((s1_sq/n1)**2/(n1-1) + (s2_sq/n2)**2/(n2-1))
    p_value = 2 * stats.t.sf(abs(t_stat), df=df)
    return t_stat, p_value

def paired_t(before: np.ndarray, after: np.ndarray) -> tuple:
    """Paired t-test. Returns (t_stat, p_value)."""
    d = after - before
    return one_sample_t(d, mu0=0.0)

# Test data
rng = np.random.default_rng(42)
ctrl = rng.normal(5.0, 1.5, 20)
trt  = rng.normal(6.0, 1.8, 20)  # true difference = 1.0
before = rng.normal(5.0, 1.0, 20)
after  = before + rng.normal(1.2, 0.5, 20)  # paired, true delta = 1.2

print("Validation: custom vs. scipy")
# One-sample
t1, p1 = one_sample_t(ctrl, mu0=4.5)
sp_t1, sp_p1 = stats.ttest_1samp(ctrl, 4.5)
print(f"One-sample:  custom t={t1:.4f} p={p1:.4f}  |  scipy t={sp_t1:.4f} p={sp_p1:.4f}")

# Welch
t2, p2 = welch_t(ctrl, trt)
sp_t2, sp_p2 = stats.ttest_ind(ctrl, trt, equal_var=False)
print(f"Welch two-sample:  custom t={t2:.4f} p={p2:.4f}  |  scipy t={sp_t2:.4f} p={sp_p2:.4f}")

# Paired
t3, p3 = paired_t(before, after)
sp_t3, sp_p3 = stats.ttest_rel(before, after)
print(f"Paired:  custom t={t3:.4f} p={p3:.4f}  |  scipy t={sp_t3:.4f} p={sp_p3:.4f}")

In [ ]:
# Cell 6.2 — Differential expression: t-test across all 500 genes
rng2 = np.random.default_rng(0)
n_genes = 500
n_ctrl, n_trt = 10, 10

# 10% of genes truly DE (effect size = 1.5)
n_de = 50
true_effects = np.zeros(n_genes)
de_genes = rng2.choice(n_genes, n_de, replace=False)
true_effects[de_genes] = 1.5

# Simulate expression
ctrl_expr = rng2.normal(5.0, 1.0, (n_genes, n_ctrl))
trt_expr = ctrl_expr + true_effects[:, np.newaxis] + rng2.normal(0, 0.3, (n_genes, n_trt))

# Run Welch t-test for each gene
t_stats = np.empty(n_genes)
p_values = np.empty(n_genes)
for g in range(n_genes):
    t_stats[g], p_values[g] = welch_t(ctrl_expr[g], trt_expr[g])

log2fc = trt_expr.mean(axis=1) - ctrl_expr.mean(axis=1)

sig_at_005 = (p_values < 0.05).sum()
print(f"Genes with p < 0.05 (uncorrected): {sig_at_005}")
print(f"True DE genes: {n_de}")
print(f"Estimated false positives: ~{(n_genes - n_de) * 0.05:.0f}")
print("→ Need multiple-testing correction (NB11)")

In [ ]:
# Cell 6.3 — Effect size: Cohen's d
def cohens_d(x: np.ndarray, y: np.ndarray) -> float:
    """Cohen's d: standardised effect size for two independent groups."""
    # Pooled standard deviation
    n1, n2 = len(x), len(y)
    s_pool = np.sqrt(((n1-1)*x.var(ddof=1) + (n2-1)*y.var(ddof=1)) / (n1+n2-2))
    return (x.mean() - y.mean()) / s_pool

d = cohens_d(ctrl, trt)
print(f"Cohen's d = {d:.3f}")
print(f"Interpretation: |d| < 0.2 small, 0.2–0.5 medium, 0.5–0.8 large, > 0.8 very large")
_, p_welch = welch_t(ctrl, trt)
print(f"p-value = {p_welch:.4f}  (p says 'significant'; d says 'how big')")

---
## Step 7 — Visualization

In [ ]:
# Cell 7.1 — Data distributions + t distributions + volcano preview
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

# Panel 1: strip plot of ctrl vs. trt
ax = axes[0]
ax.scatter(np.ones(len(ctrl)), ctrl, color="steelblue", alpha=0.6, s=40)
ax.scatter(np.ones(len(trt))*2, trt, color="tomato", alpha=0.6, s=40)
ax.hlines([ctrl.mean(), trt.mean()], [0.7, 1.7], [1.3, 2.3],
          colors=["steelblue", "tomato"], lw=2)
ax.set_xticks([1, 2]); ax.set_xticklabels(["Control", "Treatment"])
ax.set_ylabel("log₂CPM"); ax.set_title(f"Welch t-test: t={t2:.2f}, p={p2:.3f}")

# Panel 2: t distribution with rejection region
ax = axes[1]
df_approx = len(ctrl) + len(trt) - 2
t_range = np.linspace(-5, 5, 400)
ax.plot(t_range, stats.t.pdf(t_range, df=df_approx), color="black", lw=2)
crit = stats.t.ppf(0.975, df=df_approx)  # two-tailed α=0.05
ax.fill_between(t_range, stats.t.pdf(t_range, df=df_approx),
                where=t_range > crit, color="tomato", alpha=0.5, label=f"Rejection region (α=0.05)")
ax.fill_between(t_range, stats.t.pdf(t_range, df=df_approx),
                where=t_range < -crit, color="tomato", alpha=0.5)
ax.axvline(t2, color="steelblue", lw=2, label=f"t_obs = {t2:.2f}")
ax.set_xlabel("t"); ax.set_ylabel("Density")
ax.set_title("t-distribution with rejection region")
ax.legend(fontsize=8)

# Panel 3: volcano plot of 500-gene analysis
ax = axes[2]
neg_log10p = -np.log10(p_values)
sig = (p_values < 0.05) & (np.abs(log2fc) > 0.5)
ax.scatter(log2fc[~sig], neg_log10p[~sig], s=8, color="gray", alpha=0.5)
ax.scatter(log2fc[sig], neg_log10p[sig], s=15, color="tomato", alpha=0.8)
ax.axhline(-np.log10(0.05), color="black", ls="--", lw=0.8)
ax.axvline(-0.5, color="black", ls="--", lw=0.8)
ax.axvline(0.5, color="black", ls="--", lw=0.8)
ax.set_xlabel("log₂FC"); ax.set_ylabel("-log₁₀(p)")
ax.set_title("Volcano: 500 genes (needs FDR correction, NB11)")

plt.tight_layout()
plt.show()

---
## Step 8 — Exercises

1. Implement `welch_t()` from scratch without looking at Cell 6.1. Verify it against
   `scipy.stats.ttest_ind(equal_var=False)` on three different datasets.
2. For the 500-gene analysis in Cell 6.2: compute the false discovery rate at α=0.05
   without any correction. What fraction of the 'significant' genes are actually null?
3. Why does the paired t-test have more power than the unpaired test for the
   before/after data in Cell 6.1? Demonstrate by comparing the two p-values and
   explaining the difference in SE calculation.
4. A gene has mean log2CPM of 5.2 in control (n=5, sd=1.1) and 6.8 in treatment
   (n=5, sd=1.3). Compute the t-statistic and p-value by hand, then verify with scipy.
   Also compute Cohen's d.

---
## Quiz — Active Recall (Track A mock — answer cold)

1. Write the Welch t-statistic formula from memory. Define every symbol.
2. What is the difference between Student's and Welch's t-test? When should you use each?
3. What is a paired t-test? Why is it more powerful than an unpaired test on the same data?
4. What is Cohen's d? How does it differ from the t-statistic?
5. You observe p=0.04. Does this mean the probability the result is a fluke is 4%? Explain.

---
## Reflection

**Date completed:** ____________________

> *[Could you implement welch_t() from memory at a whiteboard? Could you explain paired vs. unpaired without notes?]*

---
*Next: `07_anova_basics.ipynb`*